# ISA-API Python Demo

This notebook demonstrates the Python API for working with ISA (Investigation/Study/Assay) compliant experimental metadata.

## 1. Architecture: Multi-Profile Support

The metaseed supports multiple profiles (MIAPPE, ISA, custom) using the same schema-driven architecture:

```
isa_v1.0.yaml           Single YAML file with all ISA entities
miappe_v1.1.yaml        Single YAML file with all MIAPPE entities
        |
        v
    SpecLoader          Parses YAML into ProfileSpec/EntitySpec
   (profile="isa")      Supports profile parameter
        |
        v
    ModelFactory        Dynamically creates Pydantic models
        |
        v
    get_model()         Public API: get_model(name, version, profile)
        |
        v
    User Code           Create instances, nest entities,
                        validate, serialize
```

ISA entities are based on the ISA-Tab specification and isaterms ontology.

## 2. Getting Started

In [1]:
from metaseed import get_model, validate, SpecLoader, YamlStorage, JsonStorage

## 3. Available Profiles

In [2]:
# List available profiles
loader = SpecLoader()
profiles = loader.list_profiles()
print(f"Available profiles: {profiles}")

# List versions for each profile
for profile in profiles:
    profile_loader = SpecLoader(profile=profile)
    versions = profile_loader.list_versions()
    print(f"  {profile}: versions {versions}")

Available profiles: ['isa', 'miappe']
  isa: versions ['1.0']
  miappe: versions ['1.1']


## 4. The ISA YAML Specification

All 20 ISA entities are defined in a single YAML file.

In [3]:
# Display the ISA spec file
from pathlib import Path

isa_loader = SpecLoader(profile="isa")
spec_path = isa_loader.get_profile_path("1.0")

print(f"Spec file: {spec_path.name}")
print(f"Location: {spec_path}")
print("=" * 60)
# Show first 80 lines
lines = spec_path.read_text().split('\n')[:80]
print('\n'.join(lines))
print("\n... (truncated, file continues with more entities)")

Spec file: isa_v1.0.yaml
Location: /Users/sdrwacker/workspace/miappe-tools/metaseed/src/metaseed/specs/isa_v1.0.yaml
# ISA Profile Specification v1.0
# Investigation/Study/Assay framework for describing experimental metadata
# Based on ISA-Tab specification and isaterms ontology (http://purl.org/isaterms/)

version: "1.0"
name: ISA
description: >
  Investigation/Study/Assay framework for describing life science experiments.
  The ISA framework enables experimenters to structure and share experimental
  metadata in a standard, machine-readable format.
ontology: isaterms

validation_rules:
  - name: submission_before_release
    description: Submission date should not be after public release date
    applies_to: [Investigation, Study]
    condition: public_release_date >= submission_date

entities:
  Investigation:
    ontology_term: isaterms:investigation
    description: >
      An investigation maintains metadata about the project context and links
      to one or more studies. There 

## 5. ISA Entities

ISA defines 20 entities following the ISA-Tab specification.

In [4]:
# List all ISA entities
isa_loader = SpecLoader(profile="isa")
entities = isa_loader.list_entities(version="1.0")

print(f"ISA v1.0 defines {len(entities)} entities:")
for entity in entities:
    spec = isa_loader.load_entity(entity, "1.0")
    required = [f.name for f in spec.fields if f.required]
    print(f"  {entity}: {len(spec.fields)} fields, required: {required}")

ISA v1.0 defines 20 entities:
  Assay: 10 fields, required: ['filename']
  Characteristic: 4 fields, required: ['category']
  Comment: 2 fields, required: ['name']
  DataFile: 5 fields, required: ['filename']
  Extract: 4 fields, required: ['name']
  FactorValue: 4 fields, required: []
  Investigation: 11 fields, required: ['identifier', 'title']
  LabeledExtract: 5 fields, required: ['name']
  OntologyAnnotation: 4 fields, required: ['term']
  OntologySource: 5 fields, required: ['name']
  ParameterValue: 4 fields, required: ['category']
  Person: 10 fields, required: ['last_name']
  Process: 8 fields, required: []
  Protocol: 8 fields, required: ['name']
  ProtocolParameter: 2 fields, required: ['parameter_name']
  Publication: 6 fields, required: ['title']
  Sample: 5 fields, required: ['name']
  Source: 3 fields, required: ['name']
  Study: 18 fields, required: ['identifier', 'title']
  StudyFactor: 3 fields, required: ['name']


## 6. ISA Entity Hierarchy

ISA entities follow the Investigation/Study/Assay hierarchy with a material flow:

```
Investigation
  |-- ontology_source_references: [OntologySource]
  |-- publications: [Publication]
  |-- contacts: [Person]
  |-- studies: [Study]
        |-- contacts: [Person]
        |-- publications: [Publication]
        |-- design_descriptors: [OntologyAnnotation]
        |-- protocols: [Protocol]
        |     |-- parameters: [ProtocolParameter]
        |-- factors: [StudyFactor]
        |-- sources: [Source]
        |     |-- characteristics: [Characteristic]
        |-- samples: [Sample]
        |     |-- characteristics: [Characteristic]
        |     |-- factor_values: [FactorValue]
        |     |-- derives_from: [Source]
        |-- assays: [Assay]
              |-- samples: [Sample]
              |-- data_files: [DataFile]
              |-- process_sequence: [Process]

Material Flow (experimental graph):
  Source --> Sample --> Extract --> LabeledExtract --> DataFile
     |          |           |              |
     +-- derives_from chain (provenance tracking)
```

## 7. Creating ISA Models

Use `get_model()` with `profile="isa"` to get ISA entity models.

In [5]:
import datetime

# Get ISA models
Investigation = get_model("Investigation", version="1.0", profile="isa")
Study = get_model("Study", version="1.0", profile="isa")
Assay = get_model("Assay", version="1.0", profile="isa")
Person = get_model("Person", version="1.0", profile="isa")
Publication = get_model("Publication", version="1.0", profile="isa")
Protocol = get_model("Protocol", version="1.0", profile="isa")
Source = get_model("Source", version="1.0", profile="isa")
Sample = get_model("Sample", version="1.0", profile="isa")
Extract = get_model("Extract", version="1.0", profile="isa")
LabeledExtract = get_model("LabeledExtract", version="1.0", profile="isa")
StudyFactor = get_model("StudyFactor", version="1.0", profile="isa")
DataFile = get_model("DataFile", version="1.0", profile="isa")

print("ISA models loaded successfully")
print(f"Investigation fields: {list(Investigation.model_fields.keys())}")
print(f"Extract fields: {list(Extract.model_fields.keys())}")
print(f"LabeledExtract fields: {list(LabeledExtract.model_fields.keys())}")
print(f"DataFile fields: {list(DataFile.model_fields.keys())}")

ISA models loaded successfully
Investigation fields: ['identifier', 'title', 'description', 'submission_date', 'public_release_date', 'filename', 'ontology_source_references', 'publications', 'contacts', 'studies', 'comments']
Extract fields: ['name', 'characteristics', 'derives_from', 'comments']
LabeledExtract fields: ['name', 'label', 'characteristics', 'derives_from', 'comments']
DataFile fields: ['filename', 'label', 'generated_from', 'derives_from', 'comments']


## 8. Building a Complete ISA Investigation

In [6]:
# Create an ISA Investigation for a metabolomics experiment
investigation = Investigation(
    identifier="MTBLS-2024-001",
    title="Metabolomics Analysis of Drought Stress Response in Arabidopsis",
    description="A comprehensive LC-MS metabolomics study examining metabolite "
                "changes in Arabidopsis thaliana under drought stress conditions.",
    submission_date=datetime.date(2024, 6, 1),
    public_release_date=datetime.date(2025, 1, 1),
    
    # Contacts
    contacts=[
        Person(
            last_name="Smith",
            first_name="Jane",
            email="j.smith@metabolomics-institute.org",
            affiliation="Metabolomics Research Institute",
        ),
        Person(
            last_name="Johnson",
            first_name="Robert",
            email="r.johnson@metabolomics-institute.org",
            affiliation="Metabolomics Research Institute",
        ),
    ],
    
    # Publications
    publications=[
        Publication(
            title="Metabolic signatures of drought stress in Arabidopsis",
            doi="10.1234/example.2024.001",
            author_list="Smith J, Johnson R, Williams M",
        ),
    ],
    
    # Studies
    studies=[
        Study(
            identifier="STU-DROUGHT-001",
            title="Drought Stress Metabolomics Study",
            description="Time-course metabolite profiling of drought-stressed plants",
            submission_date=datetime.date(2024, 6, 1),
            
            # Protocols
            protocols=[
                Protocol(
                    name="Plant growth",
                    description="Arabidopsis plants grown in controlled environment",
                ),
                Protocol(
                    name="Drought treatment",
                    description="Water withheld for 7 days",
                ),
                Protocol(
                    name="Metabolite extraction",
                    description="Methanol extraction of polar metabolites",
                ),
                Protocol(
                    name="LC-MS analysis",
                    description="UPLC-QTOF-MS metabolite profiling",
                ),
            ],
            
            # Factors
            factors=[
                StudyFactor(name="Treatment"),
                StudyFactor(name="Time point"),
            ],
            
            # Sources (original biological material)
            sources=[
                Source(name="Arabidopsis-Col-0-001"),
                Source(name="Arabidopsis-Col-0-002"),
                Source(name="Arabidopsis-Col-0-003"),
            ],
            
            # Samples (derived from sources)
            samples=[
                Sample(name="Control-T0-Rep1"),
                Sample(name="Control-T0-Rep2"),
                Sample(name="Drought-T7-Rep1"),
                Sample(name="Drought-T7-Rep2"),
            ],
            
            # Assays
            assays=[
                Assay(
                    filename="a_metabolite_profiling.txt",
                    technology_platform="Agilent 6550 QTOF",
                    data_files=[
                        DataFile(filename="Control-T0-Rep1.mzML", label="Raw Data File"),
                        DataFile(filename="Control-T0-Rep2.mzML", label="Raw Data File"),
                        DataFile(filename="Drought-T7-Rep1.mzML", label="Raw Data File"),
                        DataFile(filename="Drought-T7-Rep2.mzML", label="Raw Data File"),
                        DataFile(filename="processed_data.csv", label="Derived Data File"),
                    ],
                ),
            ],
        ),
    ],
)

In [7]:
# Explore the nested structure
print(f"Investigation: {investigation.identifier}")
print(f"  Title: {investigation.title}")
print(f"  Contacts: {len(investigation.contacts)}")
print(f"  Publications: {len(investigation.publications)}")
print(f"  Studies: {len(investigation.studies)}")

study = investigation.studies[0]
print(f"\n  Study: {study.identifier}")
print(f"    Protocols: {len(study.protocols)}")
print(f"    Factors: {len(study.factors)}")
print(f"    Sources: {len(study.sources)}")
print(f"    Samples: {len(study.samples)}")
print(f"    Assays: {len(study.assays)}")

assay = study.assays[0]
print(f"\n    Assay: {assay.filename}")
print(f"      Platform: {assay.technology_platform}")
print(f"      Data files: {len(assay.data_files)}")

Investigation: MTBLS-2024-001
  Title: Metabolomics Analysis of Drought Stress Response in Arabidopsis
  Contacts: 2
  Publications: 1
  Studies: 1

  Study: STU-DROUGHT-001
    Protocols: 4
    Factors: 2
    Sources: 3
    Samples: 4
    Assays: 1

    Assay: a_metabolite_profiling.txt
      Platform: Agilent 6550 QTOF
      Data files: 5


## 8b. Complete Material Flow: Source to DataFile

ISA tracks provenance through the `derives_from` chain. The complete material flow from source to data:

```
Source -> Sample -> Extract -> LabeledExtract -> DataFile
```

Each entity has a `derives_from` field linking to its predecessor in the chain.

In [8]:
# Demonstrate complete material flow: Source -> Sample -> Extract -> LabeledExtract -> DataFile

# 1. Source - the original biological material
source = Source(name="Arabidopsis-Col-0-Plant-001")

# 2. Sample - derived from the source
sample = Sample(
    name="Leaf-001",
    derives_from=[source],
)

# 3. Extract - derived from the sample (e.g., metabolite extraction)
extract = Extract(
    name="Leaf-001-MetOH-Extract",
    derives_from=[sample],
)

# 4. LabeledExtract - labeled extract for specific assay types (e.g., isotope labeling)
labeled_extract = LabeledExtract(
    name="Leaf-001-C13-Labeled",
    derives_from=[extract],
)

# 5. DataFile - the measurement data derived from the labeled extract
data_file = DataFile(
    filename="Leaf-001-C13-Labeled.mzML",
    label="Raw Data File",
    derives_from=[labeled_extract],
)

# Print the complete provenance chain
print("Complete Material Flow (derives_from chain):")
print(f"  Source: {source.name}")
print(f"    -> Sample: {sample.name}")
print(f"         -> Extract: {extract.name}")
print(f"              -> LabeledExtract: {labeled_extract.name}")
print(f"                   -> DataFile: {data_file.filename}")

# Verify the derives_from links
print(f"\nProvenance verification:")
print(f"  Sample derives from: {[s.name for s in sample.derives_from]}")
print(f"  Extract derives from: {[s.name for s in extract.derives_from]}")
print(f"  LabeledExtract derives from: {[e.name for e in labeled_extract.derives_from]}")
print(f"  DataFile derives from: {[le.name for le in data_file.derives_from]}")

Complete Material Flow (derives_from chain):
  Source: Arabidopsis-Col-0-Plant-001
    -> Sample: Leaf-001
         -> Extract: Leaf-001-MetOH-Extract
              -> LabeledExtract: Leaf-001-C13-Labeled
                   -> DataFile: Leaf-001-C13-Labeled.mzML

Provenance verification:
  Sample derives from: ['Arabidopsis-Col-0-Plant-001']
  Extract derives from: ['Leaf-001']
  LabeledExtract derives from: ['Leaf-001-MetOH-Extract']
  DataFile derives from: ['Leaf-001-C13-Labeled']


## 9. Serialization to YAML

In [9]:
import tempfile

temp_dir = Path(tempfile.mkdtemp())
yaml_storage = YamlStorage()

output_path = temp_dir / "isa_investigation.yaml"
yaml_storage.save(investigation, output_path)

print(f"Saved to: {output_path}")
print("\n" + "=" * 60)
print("YAML OUTPUT (first 100 lines):")
print("=" * 60)
lines = output_path.read_text().split('\n')[:100]
print('\n'.join(lines))
print("\n...")

Saved to: /var/folders/hr/bq19zbjx0wvbr5gmvwypgb7431fw57/T/tmpxn1ztti1/isa_investigation.yaml

YAML OUTPUT (first 100 lines):
identifier: MTBLS-2024-001
title: Metabolomics Analysis of Drought Stress Response in Arabidopsis
description: A comprehensive LC-MS metabolomics study examining metabolite changes
  in Arabidopsis thaliana under drought stress conditions.
submission_date: '2024-06-01'
public_release_date: '2025-01-01'
ontology_source_references: []
publications:
- doi: 10.1234/example.2024.001
  author_list: Smith J, Johnson R, Williams M
  title: Metabolic signatures of drought stress in Arabidopsis
  comments: []
contacts:
- last_name: Smith
  first_name: Jane
  email: j.smith@metabolomics-institute.org
  affiliation: Metabolomics Research Institute
  roles: []
  comments: []
- last_name: Johnson
  first_name: Robert
  email: r.johnson@metabolomics-institute.org
  affiliation: Metabolomics Research Institute
  roles: []
  comments: []
studies:
- identifier: STU-DROUGHT-001
  

## 10. Loading from YAML

In [10]:
# Load from YAML back into Python models
loaded_inv = yaml_storage.load(output_path, Investigation)

print(f"Loaded: {loaded_inv.title}")
print(f"Studies: {len(loaded_inv.studies)}")
print(f"First study: {loaded_inv.studies[0].title}")
print(f"Samples: {len(loaded_inv.studies[0].samples)}")

# Verify nested types are properly deserialized
print(f"\nType checks:")
print(f"  Study type: {type(loaded_inv.studies[0]).__name__}")
print(f"  Sample type: {type(loaded_inv.studies[0].samples[0]).__name__}")
print(f"  Assay type: {type(loaded_inv.studies[0].assays[0]).__name__}")

Loaded: Metabolomics Analysis of Drought Stress Response in Arabidopsis
Studies: 1
First study: Drought Stress Metabolomics Study
Samples: 4

Type checks:
  Study type: Study
  Sample type: Sample
  Assay type: Assay


## 11. JSON Schema Export

In [11]:
import json

schema = Investigation.model_json_schema()

print("ISA Investigation JSON Schema:")
print(json.dumps(schema, indent=2)[:1500] + "\n...")

ISA Investigation JSON Schema:
{
  "additionalProperties": false,
  "properties": {
    "identifier": {
      "description": "A locally unique identifier or an accession number provided by a repository",
      "title": "Identifier",
      "type": "string"
    },
    "title": {
      "description": "A concise name given to the investigation",
      "title": "Title",
      "type": "string"
    },
    "description": {
      "anyOf": [
        {
          "description": "A textual description of the investigation",
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Description"
    },
    "submission_date": {
      "anyOf": [
        {
          "description": "Date on which the investigation was reported to the repository (ISO8601)",
          "format": "date",
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Submission Date"
   

## 12. Comparing MIAPPE and ISA

Both profiles can be used in the same application.

In [12]:
# Load both profiles
miappe_loader = SpecLoader(profile="miappe")
isa_loader = SpecLoader(profile="isa")

miappe_entities = miappe_loader.list_entities("1.1")
isa_entities = isa_loader.list_entities("1.0")

print(f"MIAPPE v1.1: {len(miappe_entities)} entities")
print(f"ISA v1.0: {len(isa_entities)} entities")

# Find common entity names (case-insensitive)
miappe_lower = {e.lower() for e in miappe_entities}
isa_lower = {e.lower() for e in isa_entities}
common = miappe_lower & isa_lower

print(f"\nCommon entity names: {sorted(common)}")
print(f"MIAPPE-only: {sorted(miappe_lower - isa_lower)}")
print(f"ISA-only: {sorted(isa_lower - miappe_lower)}")

MIAPPE v1.1: 14 entities
ISA v1.0: 20 entities

Common entity names: ['datafile', 'factorvalue', 'investigation', 'person', 'sample', 'study']
MIAPPE-only: ['biologicalmaterial', 'environment', 'event', 'factor', 'location', 'materialsource', 'observationunit', 'observedvariable']
ISA-only: ['assay', 'characteristic', 'comment', 'extract', 'labeledextract', 'ontologyannotation', 'ontologysource', 'parametervalue', 'process', 'protocol', 'protocolparameter', 'publication', 'source', 'studyfactor']


## Summary

This notebook demonstrated:

1. **Multi-profile support**: Both MIAPPE and ISA profiles available
2. **ISA YAML Spec**: All 20 entities in `isa_v1.0.yaml`
3. **get_model()**: Use `profile="isa"` to get ISA models
4. **Nested Entities**: Follow ISA hierarchy (Investigation -> Study -> Assay)
5. **Serialization**: Full structure preserved in YAML/JSON
6. **Loading**: Nested entities properly deserialized as Pydantic models

The ISA profile is based on:
- ISA-Tab specification: https://isa-specs.readthedocs.io/
- isaterms ontology: http://purl.org/isaterms/